In [1]:
import requests
from bs4 import BeautifulSoup
import csv

"""commentaire
before starting web scraping we create once an virtuale environnement for this project
after we import the requests environment 
the variable url get take a link of the url of the serveur(which contain the web) that will be scraped
and after it we have the header, it is important in the sense that , it notice to the serveur that
it is not an robot but an requet and give it some information in his host, connection user-Agent the most
important thing in this header because it give it some information on the type of system on the sender computer,
the name of the web acces navigator and also all necessaire information to recognize the computer.
and finally there the variable which receive the request and we will extract the data so only what we need and put it in the csv file.
"""
""" now we want to scrape the data for multiple pages, so we will create a loop to scrape the data for multiple pages and we will change 
the page number in the url for each iteration of the loop, and we will also change the name of the csv file for each iteration of the loop 
to avoid overwriting the previous data, and we will also add a delay between each iteration of the loop to avoid getting blocked by the server. """

#donc il faut sauvegarder de temps en temps.
#et faire la boucle pour recuperer chaque page.

with open("test1.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name of the repository", "description of the repository", "programation language", "number of stars of the repository", "date of the last update of the repository"])

for page_number in range(1, 100):
    url = "https://github.com/search?q=mental+health+ai&type=repositories&p=" + str(page_number)

    header = {
        "Host": "github.com",
        "connection": "keep-alive",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
        "Referrer": "https://www/google/com/",
        "Accept-Encoding": "gzip, deflate, sdch",
        "Accept-Language": "en -US, en; q=0.8"
    }

    page = requests.get(url, headers=header)

    html = BeautifulSoup(page.content, "html.parser") #permet d'analyser le code html de la page web et de le rendre plus facile à manipuler

    #permet de trouver les titres de chaque section
    propre1 = []
    html1 = html.find_all("span", class_="search-match SearchMatchText-module__searchMatchText__n6aQc SearchMatchText-module__truncatedName__X6PYS prc-Text-Text-9mHv3")
    for element in html1:
        propre1.append(element.get_text().strip())

    #permet de trouver les descriptions de chaque section
    propre2 = []
    html2 = html.find_all("span", class_="search-match SearchMatchText-module__searchMatchText__n6aQc prc-Text-Text-9mHv3")
    for element in html2:
        propre2.append(element.get_text().strip()) #permet de supprimer les espaces blancs au début et à la fin du texte de l'élément trouvé. Cela peut être utile pour nettoyer les données extraites de la page web avant de les utiliser ou de les stocker.

    #permet de trouver le langage de programmation de chaque section
    propre3 = []
    html3 = html.find_all("span", attrs={"aria-label": True})
    for element in html3:
        propre3.append(element.get_text().strip())

    #permet de trouver le nombre d'étoiles de chaque section
    # On cherche tous les liens dont l'adresse (href) contient 'stargazers'
    stars_elements = html.find_all("a", href=lambda x: x and "stargazers" in x)
    propre4 = [s.get_text().strip() for s in stars_elements]

    # On cherche les spans qui ont un titre (title) ET qui sont dans le pied de page du résultat
    # les dates contiennent souvent "202" (pour les années 2020, 2021, etc.), donc on filtre pour ne garder que ceux-là
    dates_elements = html.find_all("span", title=True)
    propre5 = [d["title"] for d in dates_elements if "202" in d["title"]]

    data = {
        "name of the repository": propre1,
        "description of the repository": propre2,
        "programation language": propre3,
        "number of stars of the repository": propre4,
        "date of the last update of the repository": propre5
    }

    print(data)

    #permet d'écrire les données extraites dans un fichier csv les deux premiere colonnes
    with open("test1.csv", "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name of the repository", "description of the repository", "programation language", "number of stars of the repository", "date of the last update of the repository"])
        for name, description, language, stars, updated_at in zip(propre1, propre2, propre3, propre4, propre5):
            writer.writerow({
                "name of the repository": name,
                "description of the repository": description,
                "programation language": language,
                "number of stars of the repository": stars,
                "date of the last update of the repository": updated_at
            })

{'name of the repository': ['kairess/mental-health-chatbot', 'jahnavik186/AI-Mental-Health-Companion', 'PoyBoi/MindEase', 'Shivam1337/MindCrafter', 'RCTS-IIITH/AI-Assisted-Mental-Health-Screening-Application', 'dhrumilp12/Mental-Health-Companion', 'mujiyantosvc/Facial-Expression-Recognition-FER-for-Mental-Health-Detection-', 'SherazAsghar37/ai_mental_health', 'galihru/MentalHealth', 'Sanjana-Rajagopala/AI_Healthcare_Chatbot'], 'description of the repository': ['심리상담 정신건강 상담 챗봇. AI chatbot for psychology consultation.', 'An intelligent, interactive mental health companion that uses Large Language Models (LLMs) combined with RAG to provide context-aware mental', 'AI based Mental Health Counsellor / Mental Health Assistant', 'AI & ML Based Mental Health Diagnosis & Consulting App.', 'To expand the scope of existing mental health screening questionnaires by creating a Chat Bot styled testing application, allowing for en…', 'This app would provide mental health support through conversationa

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import csv
import json

"""here we use selenium to scrape Product Hunt search results across a fixed number of pages.
We stop early if the site blocks the browser or if no more products are found."""

driver_settings = webdriver.ChromeOptions()
driver_settings.add_argument("--disable-gpu")
driver_settings.add_argument("--no-sandbox")
driver_settings.add_argument("--disable-dev-shm-usage")
driver_settings.add_argument("--disable-blink-features=AutomationControlled")
driver_settings.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36")
driver_settings.add_experimental_option("excludeSwitches", ["enable-automation"])
driver_settings.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=driver_settings)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
wait = WebDriverWait(driver, 15)

all_products = []


def wait_for_cloudflare(max_attempts=3, pause_seconds=2):
    for _ in range(max_attempts):
        if "Just a moment" not in driver.title:
            return True
        time.sleep(pause_seconds)
    return "Just a moment" not in driver.title


try:
    for page_number in range(1, 24):
        search_url = f"https://www.producthunt.com/search?q=mental+health+ai&page={page_number}"
        driver.get(search_url)
        print(f"Scraping page {page_number}...")
        time.sleep(2)

        if not wait_for_cloudflare():
            print(f"Page {page_number} est bloquée par Cloudflare. J'arrête la pagination.")
            break

        try:
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "button[data-test^='spotlight-result-product-']")))
            products = driver.find_elements(By.CSS_SELECTOR, "button[data-test^='spotlight-result-product-']")

            if not products:
                print(f"Aucun produit trouvé à la page {page_number}. Fin de la pagination.")
                break

            product_count_before = len(all_products)

            for product in products:
                try:
                    name = product.find_element(By.CSS_SELECTOR, "span.text-base.font-semibold.text-dark-gray").text.strip()
                    tagline = product.find_element(By.CSS_SELECTOR, "span.text-sm.font-normal.text-light-gray").text.strip()

                    try:
                        votes = product.find_element(By.CSS_SELECTOR, "span.text-sm.font-semibold.text-brand-500").text
                    except Exception:
                        votes = "0 reviews"

                    try:
                        url = product.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                    except Exception:
                        url = ""

                    all_products.append({
                        "id": len(all_products) + 1,
                        "name": name,
                        "tagline": tagline,
                        "votes": votes.strip(),
                        "url": url
                    })
                except Exception as error:
                    print(f"Erreur lors de l'extraction d'un produit : {error}")

            products_found = len(all_products) - product_count_before
            print(f"Page {page_number} : {products_found} produits trouvés")

        except Exception as error:
            print(f"Erreur à la page {page_number} : {type(error).__name__}: {error}")
            print(f"URL actuelle : {driver.current_url}")
            print(f"Titre actuel : {driver.title}")
            break

    with open('results.json', 'w', encoding='utf-8') as json_file:
        json.dump(all_products, json_file, indent=4, ensure_ascii=False)
    print(f"\nDonnées sauvegardées dans results.json ({len(all_products)} produits)")

    if all_products:
        with open('results.csv', 'w', newline='', encoding='utf-8-sig') as csv_file:
            dict_writer = csv.DictWriter(csv_file, fieldnames=all_products[0].keys())
            dict_writer.writeheader()
            dict_writer.writerows(all_products)
    print(f"Données converties dans results.csv ({len(all_products)} produits)")

finally:
    driver.quit()
    print("Session terminée.")

Scraping page 1...
Page 1 : 10 produits trouvés
Scraping page 2...
Page 2 est bloquée par Cloudflare. J'arrête la pagination.

Données sauvegardées dans results.json (10 produits)
Données converties dans results.csv (10 produits)
Session terminée.


In [ ]:
""" dans ce code, nous utilisons Selenium pour automatiser un navigateur web (Chrome dans ce cas) 
afin de naviguer sur le site de documentation de Selenium, cliquer sur un lien spécifique et vérifier 
que les titres des pages contiennent les mots attendus. Nous configurons le navigateur pour qu'il fonctionne 
en mode headless (sans interface graphique) et nous spécifions le chemin d'accès à l'exécutable du driver Chrome 
pour permettre à Selenium de contrôler le navigateur. Enfin, nous quittons le navigateur une fois que nous avons
 terminé les vérifications. """


from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()

driver.get('https://selenium.dev/documentation')
assert 'Selenium' in driver.title

elem = driver.find_element(By.ID, 'm-documentationwebdriver')
elem.click()
assert 'WebDriver' in driver.title

driver.quit()

In [13]:
#the driver, we have to specify the path or a excutable file of the driver, but with the new version of selenium, we don't need to specify the path of the driver, selenium manager will take care of it and it will automatically download the driver if it is not already installed on the system.

Page source extraite avec succès !
